# Saving and sharing

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/widget_export.ipynb)

Every widget can save its interactive view state. Image and volume widgets can also export figures or standalone interactive HTML. `ShowEDS` adds a spectrum-image sharing path: small cubes can use single HTML with exact data, while large native EDS/EELS files use exact folder export or a smaller count-preserving downsampled single HTML export. This page uses synthetic data, so it runs anywhere.

This tutorial covers two related workflows: per-widget exports for a single view, and notebook-level sharing for collaborators, GitHub previews, and standalone HTML.

In [1]:
import numpy as np, pathlib, tempfile
from quantem.widget import Show2D, Show3D

out = pathlib.Path(tempfile.mkdtemp(prefix='quantem_widget_save_'))

# small synthetic lattice (2D) and object-phase volume (3D)
size = 256
row, col = np.mgrid[0:size, 0:size]
image = sum(np.exp(-(((row - r0) ** 2 + (col - c0) ** 2)) / (2 * 3.0 ** 2))
            for r0 in range(12, size, 24) for c0 in range(12, size, 24)).astype(np.float32)
volume = np.stack([np.roll(image, k, axis=1) for k in range(32)]).astype(np.float32)
image.shape, volume.shape

((256, 256), (32, 256, 256))

## 1. Save widget state (JSON)

`save()` writes a small, versioned JSON capturing the current view (colormap, contrast, ROIs, zoom, ...). Reload it later to restore the same view.

In [2]:
show2d = Show2D(image, pixel_size=0.2, cmap='magma', offline=True)
state_path = show2d.save(str(out / 'view.json'))
state_path = out / 'view.json'
print('state:', state_path, '-', state_path.stat().st_size, 'bytes')

state: /tmp/quantem_widget_save_mvk4ntvv/view.json - 853 bytes


## 2. Save a figure (PNG)

`save_image()` writes the colormapped image directly, or a publication figure with a title, colorbar, and scale bar when you ask for them.

In [3]:
png_path = show2d.save_image(out / 'lattice.png', scalebar=True, colorbar=True, title='Synthetic lattice')
print('figure:', png_path, '-', png_path.stat().st_size, 'bytes')


figure: /tmp/quantem_widget_save_mvk4ntvv/lattice.png - 299744 bytes


## 3. Export a standalone interactive HTML

`Show3D` and `Show3DSlices` can write a single self-contained HTML file that keeps the scrub interactive with no kernel. Two modes:

- **exact** (default): embeds float32 bytes, preserves numerical precision.
- **encoding="uint8"**: embeds the uint8 offline pack plus global min/max - ~4x smaller, visually identical after the colormap.

In [4]:
show3d = Show3D(volume, pixel_size=0.2, offline=True)
exact = show3d.export_html(out / 'volume_exact.html')
quant = show3d.export_html(out / 'volume_uint8.html', encoding="uint8")
for p in (exact, quant):
    print(f'{p.name:24s} {p.stat().st_size/1e6:6.2f} MB')

volume_exact.html         11.90 MB
volume_uint8.html          3.51 MB


## 4. Bundle exports into a zip

There is no special zip API - the exported files are ordinary files, so the standard library bundles them in a few lines.

In [5]:
import zipfile
bundle = out / 'widget_report.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in [state_path, png_path, exact, quant]:
        zf.write(p, arcname=p.name)
print('bundle:', bundle, '-', bundle.stat().st_size/1e6, 'MB')
print('contains:', zipfile.ZipFile(bundle).namelist())

bundle: /tmp/quantem_widget_save_mvk4ntvv/widget_report.zip - 1.194878 MB
contains: ['view.json', 'lattice.png', 'volume_exact.html', 'volume_uint8.html']


## 5. Share a notebook with widgets

There are three supported share artifacts, and they serve different audiences.

**Full-state notebook for collaborators.** Work in JupyterLab with a live kernel, interact with the widgets, then press Cmd+S. The notebook keeps the normal cell outputs plus `metadata.widgets`, so a compatible JupyterLab can reopen saved widget views without rerunning cells. For `ShowEDS`, the saved state includes the energy band, ROI, log toggle, overlay, contrast, and panel sizes. Small cubes stored with exact data reopen from the notebook alone; large `ShowEDS.from_emd(...)` notebooks also need the referenced data folder beside the notebook.

```bash
quantem jupyter analysis.ipynb
# interact with the widgets in JupyterLab, then Cmd+S
```

**Interactive HTML for web sharing.** Export the already-saved notebook state when you want one file that runs in a browser with no Python kernel. The HTML uses the ipywidgets HTML manager, so the controls remain interactive, but changes are browser-local and are not written back to the `.ipynb` or `.html` file.

```bash
quantem html analysis.ipynb --no-execute --out docs/
```

For a folder-backed `ShowEDS` widget, exact HTML is small but still references the data folder. Use the widget export menu or Python API for a portable public demo:

```python
widget.export_html("showeds_real_downsample4.html", downsample=4)
```

ShowEDS downsampled exports use sum reduction, so counts are preserved within the reduced pixels and energy channels.

**GitHub preview notebook.** GitHub's native notebook renderer does not hydrate widget state or execute exported HTML. Make a copy for GitHub, convert only that copy, and keep the full-state notebook unchanged for collaborators.

```bash
cp analysis.ipynb analysis_github.ipynb
quantem github analysis_github.ipynb --no-execute
```

`quantem github` removes `metadata.widgets` and widget-view MIME bundles, then embeds static JPEG snapshots of the widget UIs while preserving ordinary outputs such as prints and matplotlib images. Put `quantem html` output on GitHub Pages or another static web host if you want interactive HTML on the web; GitHub blob/raw pages are not a runnable HTML host.


```{tip}
For embedding a widget in *these docs* (or any nbconvert/Jupyter Book page), construct it with `offline=True` and let the notebook bake the state - the same mechanism `export_html` uses, but inline in the page.
```